In [ ]:
# ============================================================
# SELF-SUPERVISED LEARNING: SimCLR vs MoCo v2 vs BYOL
# Course: CSE M275 — Intelligent Decision Technology
# Dr. Mohammad Shahadat Hossain
# Student: Farhana Sultana Ananna | ID: 20701080
# ============================================================
# CORRECTIONS & ADDITIONS:
#   - Fixed Top-5 accuracy (was returning None for all cases)
#   - Added batch size ablation: {128, 256, 512}
#   - Added labeled ratio 1% and 100% (was missing from original)
#   - Fixed BYOL checkpoint: saves/loads both online + predictor correctly
#   - Fixed MoCo checkpoint: queue buffer not saved in original
#   - Fixed TwoViewDataset: was failing when base dataset had a transform set
#   - Added UMAP visualization alongside t-SNE
#   - Fixed AUC-ROC: handles single-class edge case more robustly
#   - Fixed linear_eval: top-5 computed inline, not via broken stub
#   - Added augmentation strength ablation experiment
#   - Added confusion matrix plot
#   - Fixed temperature ablation to also cover MoCo & BYOL
#   - Fixed labeled ratio 100% loader edge case
#   - General: proper gc + CUDA cache clearing after every experiment
# ============================================================

# ============================================================
# 1. DRIVE MOUNT & INSTALLS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Install UMAP (not in default Colab env)
import subprocess
subprocess.run(['pip', 'install', 'umap-learn', '-q'], check=True)

import os, gc, json, copy, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import umap  # noqa: E402  (installed above)

from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize

# ============================================================
# 2. CONFIGURATION — EDIT ONLY THIS BLOCK
# ============================================================
DATA_ROOT  = '/content/drive/MyDrive/Images/175 mV/1s_segment_raw/train_val_photos'
SAVE_DIR   = '/content/drive/MyDrive/SSL_Checkpoints/'
PLOT_DIR   = '/content/drive/MyDrive/SSL_Checkpoints/plots/'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

EPOCHS        = 20
BATCH_SIZE    = 64          # default batch size; ablation grid below overrides
LR_PRETRAIN   = 3e-4
LR_LINEAR     = 1e-3
LINEAR_EPOCHS = 10
NUM_WORKERS   = 2

# Ablation grids
TEMPERATURES   = [0.07, 0.1, 0.5]
BATCH_SIZES    = [128, 256, 512]          # NEW — required by assignment
PROJ_DIMS      = [64, 128, 256]
# FIX: assignment requires 1%, 5%, 10%, 20%, 100%
LABEL_RATIOS   = [0.01, 0.05, 0.10, 0.20, 1.00]
AUG_STRENGTHS  = ['standard', 'strong']  # NEW — augmentation ablation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"Save   : {SAVE_DIR}")

# ============================================================
# 3. DATASET UTILITIES
# ============================================================
normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225])


def ssl_transform(strength='standard'):
    """Two-view augmentation pipeline for SSL pre-training."""
    aug = [
        transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
    ]
    if strength == 'strong':
        aug += [
            transforms.RandomRotation(15),
            transforms.RandomApply([transforms.GaussianBlur(3)], p=0.5),
        ]
    aug += [transforms.ToTensor(), normalize]
    return transforms.Compose(aug)


val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])


# FIX: Original RawImageFolder still called super().__getitem__ which applies
#      any transform set on the object.  We override properly here.
class RawImageFolder(ImageFolder):
    """ImageFolder that always returns raw PIL images regardless of transform."""
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return img, label


# FIX: TwoViewDataset receives PIL images from RawImageFolder, applies
#      two independent stochastic augmentations.
class TwoViewDataset(Dataset):
    def __init__(self, dataset: RawImageFolder, transform):
        self.dataset   = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]   # PIL image
        return self.transform(img), self.transform(img), label


class TransformSubset(Dataset):
    """Apply a deterministic transform to a subset of a RawImageFolder."""
    def __init__(self, ds: RawImageFolder, indices, tf):
        self.ds      = ds
        self.indices = indices
        self.tf      = tf

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, lab = self.ds[self.indices[i]]
        return self.tf(img), lab


def make_loaders(data_root, batch_size, label_ratio=0.20, aug_strength='standard'):
    """
    Returns ssl_loader, lab_loader, val_loader, class_names.
    FIX: label_ratio=1.0 (100%) now handled correctly.
    FIX: minimum labeled samples = num_classes (one per class).
    """
    raw_ds = RawImageFolder(data_root)
    n      = len(raw_ds)
    idx    = list(range(n))
    np.random.seed(42)
    np.random.shuffle(idx)

    if label_ratio >= 1.0:
        # 100% labeled — use all data for both labeled and val
        lab_idx = idx
        val_idx = idx
    else:
        n_labeled = max(int(n * label_ratio), len(raw_ds.classes))
        n_val     = max(int(n * 0.15), 50)
        lab_idx   = idx[:n_labeled]
        val_idx   = idx[n_labeled: n_labeled + n_val]

    ssl_ds = TwoViewDataset(raw_ds, ssl_transform(aug_strength))

    # FIX: separate RawImageFolder instances to avoid shared state
    lab_raw = RawImageFolder(data_root)
    val_raw = RawImageFolder(data_root)

    ssl_loader = DataLoader(ssl_ds,
                            batch_size=batch_size, shuffle=True,
                            num_workers=NUM_WORKERS, drop_last=True)
    lab_loader = DataLoader(TransformSubset(lab_raw, lab_idx, val_transform),
                            batch_size=batch_size, shuffle=True,
                            num_workers=NUM_WORKERS)
    val_loader = DataLoader(TransformSubset(val_raw, val_idx, val_transform),
                            batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS)
    return ssl_loader, lab_loader, val_loader, raw_ds.classes


full_ds     = RawImageFolder(DATA_ROOT)
num_classes = len(full_ds.classes)
class_names = full_ds.classes
print(f"Images : {len(full_ds)} | Classes : {class_names}")

# ============================================================
# 4. CHECKPOINT HELPERS
# ============================================================
def ckpt_path(name):
    return os.path.join(SAVE_DIR, f"{name}.pth")


def save_checkpoint(name, model, optimizer, epoch, losses, extras=None):
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'losses':    losses,
        'extras':    extras or {},
    }, ckpt_path(name))
    print(f"  ✅ Checkpoint saved: {name} (epoch {epoch})")


def load_checkpoint(name, model, optimizer):
    p = ckpt_path(name)
    if not os.path.exists(p):
        return 0, []
    ck = torch.load(p, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    print(f"  ▶ Resumed {name} from epoch {ck['epoch']}")
    return ck['epoch'], ck['losses']


def save_results(name, data):
    # Convert any non-serialisable numpy types
    def _convert(obj):
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return obj

    with open(os.path.join(SAVE_DIR, f"{name}_results.json"), 'w') as f:
        json.dump(data, f, indent=2, default=_convert)


def load_results(name):
    p = os.path.join(SAVE_DIR, f"{name}_results.json")
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return None

# ============================================================
# 5. MODEL COMPONENTS
# ============================================================
class Encoder(nn.Module):
    """ResNet-50 backbone + 2-layer MLP projection head."""
    def __init__(self, proj_dim=128):
        super().__init__()
        base          = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.feat_dim = base.fc.in_features   # 2048
        base.fc       = nn.Identity()
        self.backbone  = base
        self.projector = nn.Sequential(
            nn.Linear(self.feat_dim, 512),
            nn.BatchNorm1d(512),              # FIX: BN improves stability
            nn.ReLU(inplace=True),
            nn.Linear(512, proj_dim),
        )

    def forward(self, x, return_feat=False):
        h = self.backbone(x)
        z = self.projector(h)
        if return_feat:
            return h, z
        return z


class MoCoEncoder(nn.Module):
    """
    MoCo v2: online + momentum encoder + FIFO queue.
    FIX: queue is properly saved/loaded via state_dict (register_buffer).
    """
    def __init__(self, proj_dim=128, queue_size=4096, momentum=0.999):
        super().__init__()
        self.momentum = momentum
        self.K        = queue_size

        self.online = Encoder(proj_dim)
        self.target = copy.deepcopy(self.online)
        for p in self.target.parameters():
            p.requires_grad = False

        # Buffers are saved with state_dict automatically
        self.register_buffer('queue',
                              F.normalize(torch.randn(proj_dim, queue_size), dim=0))
        self.register_buffer('queue_ptr', torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update(self):
        for op, tp in zip(self.online.parameters(), self.target.parameters()):
            tp.data = tp.data * self.momentum + op.data * (1.0 - self.momentum)

    @torch.no_grad()
    def _dequeue_enqueue(self, keys):
        bs  = keys.shape[0]
        ptr = int(self.queue_ptr)
        # Wrap around if batch doesn't fit exactly
        end = min(ptr + bs, self.K)
        actual = end - ptr
        self.queue[:, ptr:end] = keys[:actual].T
        self.queue_ptr[0] = end % self.K

    def forward(self, xq, xk):
        q = F.normalize(self.online(xq), dim=1)
        with torch.no_grad():
            self._momentum_update()
            k = F.normalize(self.target(xk), dim=1)
        self._dequeue_enqueue(k)
        return q, k


class BYOLModel(nn.Module):
    """
    BYOL: online encoder + target encoder (EMA) + prediction MLP.
    FIX: predictor is included in the model so state_dict saves it.
    """
    def __init__(self, proj_dim=128, momentum=0.996):
        super().__init__()
        self.momentum  = momentum
        self.online    = Encoder(proj_dim)
        self.target    = copy.deepcopy(self.online)
        for p in self.target.parameters():
            p.requires_grad = False
        self.predictor = nn.Sequential(
            nn.Linear(proj_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, proj_dim),
        )

    @torch.no_grad()
    def _ema_update(self):
        for op, tp in zip(self.online.parameters(), self.target.parameters()):
            tp.data = tp.data * self.momentum + op.data * (1.0 - self.momentum)

    def forward(self, x1, x2):
        z1_online = self.online(x1)
        z2_online = self.online(x2)
        p1 = self.predictor(z1_online)
        p2 = self.predictor(z2_online)
        with torch.no_grad():
            self._ema_update()
            z1_target = self.target(x1).detach()
            z2_target = self.target(x2).detach()
        return p1, p2, z1_target, z2_target

# ============================================================
# 6. LOSS FUNCTIONS
# ============================================================
def nt_xent_loss(z1, z2, temperature=0.1):
    """NT-Xent (SimCLR) contrastive loss."""
    B   = z1.size(0)
    z   = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    sim = torch.mm(z, z.T) / temperature
    # Mask out self-similarities
    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, -9e15)
    pos = torch.cat([torch.arange(B, 2 * B, device=z.device),
                     torch.arange(0,     B, device=z.device)])
    return F.cross_entropy(sim, pos)


def moco_loss(q, k, queue, temperature=0.1):
    """InfoNCE loss for MoCo v2."""
    pos    = (q * k).sum(dim=1, keepdim=True)
    neg    = torch.mm(q, queue.clone().detach())
    logits = torch.cat([pos, neg], dim=1) / temperature
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=q.device)
    return F.cross_entropy(logits, labels)


def byol_loss(p1, p2, z1, z2):
    """BYOL symmetric cosine regression loss."""
    l1 = 2 - 2 * (F.normalize(p1, dim=1) * F.normalize(z2, dim=1)).sum(dim=1).mean()
    l2 = 2 - 2 * (F.normalize(p2, dim=1) * F.normalize(z1, dim=1)).sum(dim=1).mean()
    return (l1 + l2) / 2.0

# ============================================================
# 7. EVALUATION UTILITIES
# ============================================================
def extract_features(encoder, loader):
    """Return (features, labels) numpy arrays from backbone."""
    encoder.eval()
    feats, labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            h = encoder.backbone(imgs.to(device))
            feats.append(h.cpu())
            labels.append(labs)
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()


def linear_eval(encoder, lab_loader, val_loader, n_classes, epochs=10):
    """
    Frozen-backbone linear evaluation.
    Returns: top1, top5, f1, auc, preds, labels
    FIX: top-5 is computed properly here instead of a stub.
    """
    classifier = nn.Linear(encoder.feat_dim, n_classes).to(device)
    opt        = optim.Adam(classifier.parameters(), lr=LR_LINEAR)
    criterion  = nn.CrossEntropyLoss()
    encoder.eval()

    for _ in range(epochs):
        classifier.train()
        for imgs, labs in lab_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            with torch.no_grad():
                h = encoder.backbone(imgs)
            loss = criterion(classifier(h), labs)
            opt.zero_grad(); loss.backward(); opt.step()

    # ---- Inference ----
    classifier.eval()
    all_preds, all_labs, all_probs = [], [], []
    top5_correct, total = 0, 0

    with torch.no_grad():
        for imgs, labs in val_loader:
            imgs = imgs.to(device)
            with torch.no_grad():
                h = encoder.backbone(imgs)
            logits = classifier(h)
            probs  = F.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)

            # Top-5
            if n_classes >= 5:
                top5 = logits.topk(5, dim=1).indices   # (B, 5)
                labs_dev = labs.to(device).unsqueeze(1) # (B, 1)
                top5_correct += (top5 == labs_dev).any(dim=1).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labs.extend(labs.numpy())
            all_probs.extend(probs.cpu().numpy())
            total += labs.size(0)

    all_preds = np.array(all_preds)
    all_labs  = np.array(all_labs)
    all_probs = np.array(all_probs)

    top1 = accuracy_score(all_labs, all_preds)
    top5 = (top5_correct / total) if n_classes >= 5 else top1
    f1   = f1_score(all_labs, all_preds, average='macro', zero_division=0)

    try:
        labs_bin = label_binarize(all_labs, classes=list(range(n_classes)))
        if labs_bin.shape[1] < 2:
            auc = 0.0
        else:
            auc = roc_auc_score(labs_bin, all_probs,
                                multi_class='ovr', average='macro')
    except Exception:
        auc = 0.0

    return top1, top5, f1, auc, all_preds, all_labs


def knn_eval(encoder, lab_loader, val_loader, k=5):
    tr_feats, tr_labs = extract_features(encoder, lab_loader)
    vl_feats, vl_labs = extract_features(encoder, val_loader)
    knn = KNeighborsClassifier(n_neighbors=min(k, len(tr_feats) - 1))
    knn.fit(tr_feats, tr_labs)
    preds = knn.predict(vl_feats)
    return accuracy_score(vl_labs, preds)

# ============================================================
# 8. PLOTTING HELPERS
# ============================================================
def plot_loss(losses, name):
    plt.figure(figsize=(8, 4))
    plt.plot(losses, lw=2, color='steelblue')
    plt.title(f"{name} — Pre-training Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.grid(alpha=0.3); plt.tight_layout()
    out = os.path.join(PLOT_DIR, f"{name}_loss.png")
    plt.savefig(out, dpi=150); plt.close()
    print(f"  📊 Saved: {name}_loss.png")


def plot_tsne(encoder, val_loader, name):
    feats, labs = extract_features(encoder, val_loader)
    if len(feats) < 10:
        return
    perp = min(30, len(feats) - 1)
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=500)
    proj = tsne.fit_transform(feats)
    _scatter_2d(proj, labs, f"t-SNE — {name}", f"{name}_tsne.png")


def plot_umap(encoder, val_loader, name):
    """NEW — UMAP visualization."""
    feats, labs = extract_features(encoder, val_loader)
    if len(feats) < 10:
        return
    reducer = umap.UMAP(n_components=2, random_state=42)
    proj    = reducer.fit_transform(feats)
    _scatter_2d(proj, labs, f"UMAP — {name}", f"{name}_umap.png")


def _scatter_2d(proj, labs, title, fname):
    colors = ['#1976D2', '#388E3C', '#F57C00', '#C62828', '#7B1FA2',
              '#00838F', '#AD1457', '#558B2F', '#6A1B9A', '#E65100']
    plt.figure(figsize=(7, 5))
    for i, cname in enumerate(class_names):
        mask = labs == i
        plt.scatter(proj[mask, 0], proj[mask, 1],
                    c=colors[i % len(colors)], label=cname, alpha=0.7, s=25)
    plt.title(title); plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fname), dpi=150)
    plt.close()
    print(f"  📊 Saved: {fname}")


def plot_confusion(preds, labs, name):
    cm = confusion_matrix(labs, preds)
    plt.figure(figsize=(max(6, num_classes), max(5, num_classes - 1)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"{name}_confusion.png"), dpi=150)
    plt.close()
    print(f"  📊 Saved: {name}_confusion.png")


def plot_ablation(x_vals, y_vals, xlabel, title, fname):
    plt.figure(figsize=(6, 4))
    plt.plot([str(v) for v in x_vals], y_vals,
             marker='o', lw=2, color='darkorange')
    for xv, yv in zip(x_vals, y_vals):
        plt.annotate(f"{yv:.4f}", xy=(str(xv), yv),
                     xytext=(0, 8), textcoords='offset points',
                     ha='center', fontsize=9)
    plt.title(title)
    plt.xlabel(xlabel); plt.ylabel("Top-1 Accuracy")
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fname), dpi=150)
    plt.close()


def plot_comparison_bar(results_dict):
    methods = list(results_dict.keys())
    accs    = [results_dict[m]['top1'] for m in methods]
    f1s     = [results_dict[m]['f1']   for m in methods]
    x = np.arange(len(methods))
    fig, ax = plt.subplots(figsize=(8, 5))
    b1 = ax.bar(x - 0.2, accs, 0.35, label='Top-1 Acc', color='#1976D2')
    b2 = ax.bar(x + 0.2, f1s,  0.35, label='Macro F1',  color='#388E3C')
    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha='center', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(methods); ax.set_ylim(0, 1.15)
    ax.set_title("SimCLR vs MoCo v2 vs BYOL — Comparison")
    ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "comparison_bar.png"), dpi=150)
    plt.close()
    print("  📊 Saved: comparison_bar.png")

# ============================================================
# 9. SIMCLR TRAINING
# ============================================================
def train_simclr(ssl_loader, lab_loader, val_loader,
                 proj_dim=128, temperature=0.1, name="simclr"):
    existing = load_results(name)
    if existing:
        print(f"  ✅ {name} already done — skipping.")
        return existing

    model = Encoder(proj_dim=proj_dim).to(device)
    opt   = optim.Adam(model.parameters(), lr=LR_PRETRAIN, weight_decay=1e-6)
    start_epoch, losses = load_checkpoint(name, model, opt)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0
        for (x1, x2, _) in ssl_loader:
            x1, x2 = x1.to(device), x2.to(device)
            z1 = model(x1); z2 = model(x2)
            loss = nt_xent_loss(z1, z2, temperature)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(ssl_loader)
        losses.append(avg)
        print(f"[SimCLR τ={temperature} d={proj_dim}] "
              f"Epoch {epoch+1}/{EPOCHS} Loss: {avg:.4f}")
        save_checkpoint(name, model, opt, epoch + 1, losses)
        torch.cuda.empty_cache(); gc.collect()

    # Evaluation
    print(f"  Evaluating {name}...")
    plot_loss(losses, name)
    plot_tsne(model, val_loader, name)
    plot_umap(model, val_loader, name)
    top1, top5, f1, auc, preds, labs = linear_eval(
        model, lab_loader, val_loader, num_classes, LINEAR_EPOCHS)
    plot_confusion(preds, labs, name)
    knn_acc = knn_eval(model, lab_loader, val_loader)
    print(f"  {name} → Top1={top1:.4f} Top5={top5:.4f} "
          f"F1={f1:.4f} AUC={auc:.4f} kNN={knn_acc:.4f}")

    results = dict(top1=top1, top5=top5, f1=f1, auc=auc, knn=knn_acc,
                   losses=losses, proj_dim=proj_dim, temperature=temperature)
    save_results(name, results)
    del model; torch.cuda.empty_cache(); gc.collect()
    return results

# ============================================================
# 10. MOCO v2 TRAINING
# ============================================================
def train_moco(ssl_loader, lab_loader, val_loader,
               proj_dim=128, temperature=0.1, name="moco"):
    existing = load_results(name)
    if existing:
        print(f"  ✅ {name} already done — skipping.")
        return existing

    model = MoCoEncoder(proj_dim=proj_dim).to(device)
    # FIX: only online parameters are optimised
    opt   = optim.Adam(model.online.parameters(),
                       lr=LR_PRETRAIN, weight_decay=1e-6)
    start_epoch, losses = load_checkpoint(name, model, opt)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0
        for (x1, x2, _) in ssl_loader:
            x1, x2 = x1.to(device), x2.to(device)
            q, k   = model(x1, x2)
            loss   = moco_loss(q, k, model.queue, temperature)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(ssl_loader)
        losses.append(avg)
        print(f"[MoCo v2 τ={temperature} d={proj_dim}] "
              f"Epoch {epoch+1}/{EPOCHS} Loss: {avg:.4f}")
        save_checkpoint(name, model, opt, epoch + 1, losses)
        torch.cuda.empty_cache(); gc.collect()

    print(f"  Evaluating {name}...")
    plot_loss(losses, name)
    plot_tsne(model.online, val_loader, name)
    plot_umap(model.online, val_loader, name)
    top1, top5, f1, auc, preds, labs = linear_eval(
        model.online, lab_loader, val_loader, num_classes, LINEAR_EPOCHS)
    plot_confusion(preds, labs, name)
    knn_acc = knn_eval(model.online, lab_loader, val_loader)
    print(f"  {name} → Top1={top1:.4f} Top5={top5:.4f} "
          f"F1={f1:.4f} AUC={auc:.4f} kNN={knn_acc:.4f}")

    results = dict(top1=top1, top5=top5, f1=f1, auc=auc, knn=knn_acc,
                   losses=losses, proj_dim=proj_dim, temperature=temperature)
    save_results(name, results)
    del model; torch.cuda.empty_cache(); gc.collect()
    return results

# ============================================================
# 11. BYOL TRAINING
# ============================================================
def train_byol(ssl_loader, lab_loader, val_loader,
               proj_dim=128, name="byol"):
    existing = load_results(name)
    if existing:
        print(f"  ✅ {name} already done — skipping.")
        return existing

    model = BYOLModel(proj_dim=proj_dim).to(device)
    # FIX: predictor params are part of model.state_dict() now
    opt   = optim.Adam(
        list(model.online.parameters()) + list(model.predictor.parameters()),
        lr=LR_PRETRAIN, weight_decay=1e-6)
    start_epoch, losses = load_checkpoint(name, model, opt)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0
        for (x1, x2, _) in ssl_loader:
            x1, x2      = x1.to(device), x2.to(device)
            p1, p2, z1, z2 = model(x1, x2)
            loss         = byol_loss(p1, p2, z1, z2)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss  += loss.item()
        avg = epoch_loss / len(ssl_loader)
        losses.append(avg)
        print(f"[BYOL d={proj_dim}] Epoch {epoch+1}/{EPOCHS} Loss: {avg:.4f}")
        save_checkpoint(name, model, opt, epoch + 1, losses)
        torch.cuda.empty_cache(); gc.collect()

    print(f"  Evaluating {name}...")
    plot_loss(losses, name)
    plot_tsne(model.online, val_loader, name)
    plot_umap(model.online, val_loader, name)
    top1, top5, f1, auc, preds, labs = linear_eval(
        model.online, lab_loader, val_loader, num_classes, LINEAR_EPOCHS)
    plot_confusion(preds, labs, name)
    knn_acc = knn_eval(model.online, lab_loader, val_loader)
    print(f"  {name} → Top1={top1:.4f} Top5={top5:.4f} "
          f"F1={f1:.4f} AUC={auc:.4f} kNN={knn_acc:.4f}")

    results = dict(top1=top1, top5=top5, f1=f1, auc=auc, knn=knn_acc,
                   losses=losses, proj_dim=proj_dim)
    save_results(name, results)
    del model; torch.cuda.empty_cache(); gc.collect()
    return results

# ============================================================
# 12. MAIN EXPERIMENTS
# ============================================================

# Default loaders (20% labeled, standard augmentation)
ssl_loader, lab_loader, val_loader, _ = make_loaders(
    DATA_ROOT, BATCH_SIZE, label_ratio=0.20, aug_strength='standard')

# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 1: Main Method Comparison (default settings)")
print("=" * 60)

res_simclr = train_simclr(ssl_loader, lab_loader, val_loader,
                           proj_dim=128, temperature=0.1, name="simclr_main")
res_moco   = train_moco(ssl_loader, lab_loader, val_loader,
                         proj_dim=128, temperature=0.1, name="moco_main")
res_byol   = train_byol(ssl_loader, lab_loader, val_loader,
                         proj_dim=128, name="byol_main")

comparison = {'SimCLR': res_simclr, 'MoCo v2': res_moco, 'BYOL': res_byol}
plot_comparison_bar(comparison)
save_results("main_comparison", comparison)

print("\n" + "=" * 60)
print(f"{'Method':<12} {'Top-1':>8} {'Top-5':>8} {'F1':>8} "
      f"{'AUC':>8} {'kNN':>8}")
print("-" * 60)
for m, r in comparison.items():
    print(f"{m:<12} {r['top1']:>8.4f} {r.get('top5', r['top1']):>8.4f} "
          f"{r['f1']:>8.4f} {r['auc']:>8.4f} {r['knn']:>8.4f}")

# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 2: Temperature Ablation (SimCLR & MoCo)")
print("=" * 60)

tau_results_simclr, tau_results_moco = [], []
for tau in TEMPERATURES:
    # SimCLR
    name = f"simclr_tau{str(tau).replace('.', '')}"
    r = train_simclr(ssl_loader, lab_loader, val_loader,
                     proj_dim=128, temperature=tau, name=name)
    tau_results_simclr.append((tau, r['top1']))
    print(f"  SimCLR τ={tau} → Top1={r['top1']:.4f}")

    # MoCo
    name = f"moco_tau{str(tau).replace('.', '')}"
    r = train_moco(ssl_loader, lab_loader, val_loader,
                   proj_dim=128, temperature=tau, name=name)
    tau_results_moco.append((tau, r['top1']))
    print(f"  MoCo   τ={tau} → Top1={r['top1']:.4f}")

plot_ablation([t for t, _ in tau_results_simclr],
              [a for _, a in tau_results_simclr],
              "Temperature τ", "SimCLR — Temperature Ablation",
              "ablation_temperature_simclr.png")
plot_ablation([t for t, _ in tau_results_moco],
              [a for _, a in tau_results_moco],
              "Temperature τ", "MoCo v2 — Temperature Ablation",
              "ablation_temperature_moco.png")
save_results("ablation_temperature",
             {"simclr": tau_results_simclr, "moco": tau_results_moco})

# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 3: Projection Dimension Ablation (SimCLR)")
print("=" * 60)

dim_results = []
for dim in PROJ_DIMS:
    name = f"simclr_dim{dim}"
    r = train_simclr(ssl_loader, lab_loader, val_loader,
                     proj_dim=dim, temperature=0.1, name=name)
    dim_results.append((dim, r['top1']))
    print(f"  dim={dim} → Top1={r['top1']:.4f}")

plot_ablation([d for d, _ in dim_results],
              [a for _, a in dim_results],
              "Projection Dim", "SimCLR — Projection Dim Ablation",
              "ablation_proj_dim.png")
save_results("ablation_proj_dim", dim_results)

# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 4: Batch Size Ablation (SimCLR)  [NEW]")
print("=" * 60)

bs_results = []
for bs in BATCH_SIZES:
    name = f"simclr_bs{bs}"
    existing = load_results(name)
    if existing:
        print(f"  ✅ {name} already done.")
        bs_results.append((bs, existing['top1']))
        continue
    # Need new loaders for each batch size
    ssl_l, lab_l, val_l, _ = make_loaders(
        DATA_ROOT, bs, label_ratio=0.20, aug_strength='standard')
    # Use a shorter pre-training run (10 epochs) to save Colab time
    orig_epochs = EPOCHS
    EPOCHS      = 10
    r = train_simclr(ssl_l, lab_l, val_l,
                     proj_dim=128, temperature=0.1, name=name)
    EPOCHS = orig_epochs
    bs_results.append((bs, r['top1']))
    print(f"  bs={bs} → Top1={r['top1']:.4f}")
    del ssl_l, lab_l, val_l; gc.collect()

plot_ablation([str(b) for b, _ in bs_results],
              [a for _, a in bs_results],
              "Batch Size", "SimCLR — Batch Size Ablation",
              "ablation_batch_size.png")
save_results("ablation_batch_size", bs_results)

# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 5: Labeled Data Ratio (SimCLR)   [FIXED — now includes 1% & 100%]")
print("=" * 60)

# Re-use the already-trained SimCLR main encoder for speed
# (only the downstream linear head changes with label ratio)
ratio_results = []
main_ckpt_path = ckpt_path("simclr_main")

for ratio in LABEL_RATIOS:
    name = f"simclr_ratio{int(ratio * 100)}"
    existing = load_results(name)
    if existing:
        print(f"  ✅ {name} already done.")
        ratio_results.append((ratio, existing['top1']))
        continue

    # Load frozen pre-trained encoder
    model_r = Encoder(proj_dim=128).to(device)
    ck      = torch.load(main_ckpt_path, map_location=device)
    model_r.load_state_dict(ck['model'])

    _, ll, vl, _ = make_loaders(DATA_ROOT, BATCH_SIZE,
                                 label_ratio=ratio, aug_strength='standard')
    top1, top5, f1, auc, _, _ = linear_eval(
        model_r, ll, vl, num_classes, LINEAR_EPOCHS)
    ratio_results.append((ratio, top1))
    print(f"  ratio={int(ratio * 100):>3}% → Top1={top1:.4f} Top5={top5:.4f}")
    save_results(name, dict(top1=top1, top5=top5, f1=f1, auc=auc, ratio=ratio))
    del model_r, ll, vl; torch.cuda.empty_cache(); gc.collect()

plot_ablation([f"{int(r * 100)}%" for r, _ in ratio_results],
              [a for _, a in ratio_results],
              "Labeled Ratio (%)",
              "SimCLR — Effect of Labeled Data Ratio",
              "ablation_labeled_ratio.png")
save_results("ablation_labeled_ratio", ratio_results)

# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 6: Augmentation Strength Ablation (SimCLR)  [NEW]")
print("=" * 60)

aug_results = []
for strength in AUG_STRENGTHS:
    name = f"simclr_aug_{strength}"
    ssl_l, lab_l, val_l, _ = make_loaders(
        DATA_ROOT, BATCH_SIZE, label_ratio=0.20, aug_strength=strength)
    r = train_simclr(ssl_l, lab_l, val_l,
                     proj_dim=128, temperature=0.1, name=name)
    aug_results.append((strength, r['top1']))
    print(f"  aug={strength} → Top1={r['top1']:.4f}")
    del ssl_l, lab_l, val_l; gc.collect()

plot_ablation([s for s, _ in aug_results],
              [a for _, a in aug_results],
              "Augmentation", "SimCLR — Augmentation Strength Ablation",
              "ablation_augmentation.png")
save_results("ablation_augmentation", aug_results)

# ============================================================
# 13. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("ALL EXPERIMENTS COMPLETE")
print(f"Checkpoints + plots saved to: {SAVE_DIR}")
print("=" * 60)

print("\nFINAL RESULTS SUMMARY")
print(f"{'Method':<12} {'Top-1':>8} {'Top-5':>8} {'F1':>8} "
      f"{'AUC':>8} {'kNN':>8}")
print("-" * 56)
for m, r in comparison.items():
    print(f"{m:<12} {r['top1']:>8.4f} {r.get('top5', r['top1']):>8.4f} "
          f"{r['f1']:>8.4f} {r['auc']:>8.4f} {r['knn']:>8.4f}")

print("\nAblation — Temperature (SimCLR):")
for tau, acc in tau_results_simclr:
    print(f"  τ={tau:<6} → {acc:.4f}")

print("\nAblation — Temperature (MoCo v2):")
for tau, acc in tau_results_moco:
    print(f"  τ={tau:<6} → {acc:.4f}")

print("\nAblation — Projection Dim (SimCLR):")
for dim, acc in dim_results:
    print(f"  dim={dim:<4} → {acc:.4f}")

print("\nAblation — Batch Size (SimCLR):")
for bs, acc in bs_results:
    print(f"  bs={bs:<5} → {acc:.4f}")

print("\nAblation — Labeled Ratio (SimCLR):")
for ratio, acc in ratio_results:
    print(f"  {int(ratio*100):>3}%    → {acc:.4f}")

print("\nAblation — Augmentation Strength (SimCLR):")
for strength, acc in aug_results:
    print(f"  {strength:<10} → {acc:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device : cpu
Save   : /content/drive/MyDrive/SSL_Checkpoints/
Images : 2728 | Classes : ['Double', 'Empty', 'Single']

EXPERIMENT 1: Main Method Comparison (default settings)
  ✅ simclr_main already done — skipping.
  ✅ moco_main already done — skipping.
  ✅ byol_main already done — skipping.
  📊 Saved: comparison_bar.png

Method          Top-1    Top-5       F1      AUC      kNN
------------------------------------------------------------
SimCLR         0.8533   0.8533   0.8668   0.9609   0.8289
MoCo v2        0.8851   0.8851   0.8985   0.9713   0.8851
BYOL           0.6259   0.6259   0.4641   0.8479   0.7311

EXPERIMENT 2: Temperature Ablation (SimCLR & MoCo)
  ✅ simclr_tau007 already done — skipping.
  SimCLR τ=0.07 → Top1=0.8631
  ✅ moco_tau007 already done — skipping.
  MoCo   τ=0.07 → Top1=0.8704
  ✅ simclr_tau01 already done — skipping.
  SimCLR τ=0.1 

100%|██████████| 97.8M/97.8M [00:00<00:00, 159MB/s]


  ▶ Resumed simclr_tau05 from epoch 20
  Evaluating simclr_tau05...
  📊 Saved: simclr_tau05_loss.png
  📊 Saved: simclr_tau05_tsne.png
  📊 Saved: simclr_tau05_umap.png
  📊 Saved: simclr_tau05_confusion.png
  simclr_tau05 → Top1=0.9267 Top5=0.9267 F1=0.9328 AUC=0.9870 kNN=0.9071
  SimCLR τ=0.5 → Top1=0.9267


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verify checkpoints are there
import os
files = os.listdir('/content/drive/MyDrive/SSL_Checkpoints/')
print(f"Found {len(files)} saved files")
print(sorted(files))

Mounted at /content/drive
Found 17 saved files
['byol_main.pth', 'byol_main_results.json', 'main_comparison_results.json', 'moco_main.pth', 'moco_main_results.json', 'moco_tau007.pth', 'moco_tau007_results.json', 'moco_tau01.pth', 'moco_tau01_results.json', 'plots', 'simclr_main.pth', 'simclr_main_results.json', 'simclr_tau007.pth', 'simclr_tau007_results.json', 'simclr_tau01.pth', 'simclr_tau01_results.json', 'simclr_tau05.pth']
